# Smart Fraud Detection Pipeline
### Dashboard Data Preparation

This notebook prepares business-ready datasets from the Gold layer for visualization and reporting.
The transformation logic is intentionally kept separate from the dashboard so that the Gold layer remains the single trusted source for analytical outputs.

### Dashboard Outputs

- Transaction-level fraud dataset
- Daily fraud trend
- International vs domestic fraud analysis
- Fraud summary KPIs
- Account-level fraud analysis
- Fraud-type analysis


In [0]:
from pyspark.sql import functions as F

df_fraud = spark.table(
    "fraud_detection.gold.fraud_transactions"
)

df_account = spark.table(
    "fraud_detection.gold.account_fraud_analysis"
)

df_type = spark.table(
    "fraud_detection.gold.fraud_type_analysis"
)

df_kpi = spark.table(
    "fraud_detection.gold.executive_kpis"
)

#### Create dashboard transaction view

In [0]:
dashboard_transactions = df_fraud.select(
    "txn_id",
    "account_id",
    "customer_name",
    "account_type",
    "txn_date",
    "amount",
    "merchant",
    "city",
    "is_international",
    "fraud_type",
    "fraud_status",
    "fraud_flag",
    "fraud_amount"
)

display(dashboard_transactions)

txn_id,account_id,customer_name,account_type,txn_date,amount,merchant,city,is_international,fraud_type,fraud_status,fraud_flag,fraud_amount
TXN-000001,ACC-00056,null,null,2026-01-09,915931.22,UNKNOWN,Singapore,true,ACCOUNT_TAKEOVER,fraud,1,915931.22
TXN-000002,ACC-00025,Hemant Jain,NRI,2026-01-11,23892.36,BookMyShow,Bangalore,false,null,normal,0,0.0
TXN-000003,ACC-00029,Tanuja Nair,SALARY,2026-02-03,15277.03,Swiggy,New York,true,MONEY_LAUNDERING,fraud,1,15277.03
TXN-000004,ACC-00007,Prakash Rao,NRI,2026-03-05,5498.74,ATM,Delhi,false,null,normal,0,0.0
TXN-000005,ACC-00029,Tanuja Nair,SALARY,2026-02-23,919.78,Flipkart,Chennai,false,MONEY_LAUNDERING,fraud,1,919.78
TXN-000006,ACC-00047,Pawan Arora,SAVINGS,2026-02-09,22417.98,Swiggy,Mumbai,false,null,normal,0,0.0
TXN-000007,ACC-00003,Anil Reddy,CURRENT,2026-02-22,3832.61,BigBasket,Singapore,true,null,normal,0,0.0
TXN-000008,ACC-00037,Naveen Chadha,SALARY,2026-03-01,11979.58,PhonePe,Delhi,false,null,normal,0,0.0
TXN-000009,ACC-00035,Tarun Bose,SAVINGS,2026-01-08,23856.18,Unknown_Merchant,New York,true,null,normal,0,0.0
TXN-000010,ACC-00008,Lata Mishra,SALARY,2026-03-19,8369.32,BookMyShow,Singapore,true,null,normal,0,0.0


In [0]:
(
    dashboard_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fraud_detection.gold.dashboard_transactions"
    )
)

#### Create daily fraud trend

In [0]:
daily_fraud = (
    df_fraud
    .groupBy("txn_date")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum("fraud_flag").alias(
            "fraud_transactions"
        ),

        F.sum("fraud_amount").alias(
            "fraud_amount"
        )
    )
    .withColumn(
        "fraud_rate",
        F.round(
            F.col("fraud_transactions") /
            F.col("total_transactions") * 100,
            2
        )
    )
    .orderBy("txn_date")
)

display(daily_fraud)

txn_date,total_transactions,fraud_transactions,fraud_amount,fraud_rate
2026-01-01,1,0,0.0,0.0
2026-01-02,4,2,31339.8,50.0
2026-01-03,1,0,0.0,0.0
2026-01-04,1,0,0.0,0.0
2026-01-05,4,0,0.0,0.0
2026-01-06,3,0,0.0,0.0
2026-01-07,3,0,0.0,0.0
2026-01-08,4,0,0.0,0.0
2026-01-09,2,1,915931.22,50.0
2026-01-10,1,0,0.0,0.0


In [0]:
(
    daily_fraud
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fraud_detection.gold.daily_fraud_trend"
    )
)

#### Create international fraud analysis

In [0]:
international_analysis = (
    df_fraud
    .groupBy("is_international")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("fraud_flag").alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .withColumn(
        "fraud_rate",
        F.round(
            F.col("fraud_transactions") /
            F.col("total_transactions") * 100,
            2
        )
    )
)

display(international_analysis)

is_international,total_transactions,fraud_transactions,fraud_amount,fraud_rate
true,119,16,1878316.8200000003,13.45
false,81,10,1929315.29,12.35


In [0]:
(
    international_analysis
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "fraud_detection.gold.international_fraud_analysis"
    )
)